# 04 — Logging Utils (`core/logging_utils.py`)
Production-grade logging with:
- **Console** handler (human-readable, timestamped)
- **File** handler (`logs/copilot.log`) in structured JSON
- **Decorators**: `@with_retry`, `@timed`
- **Custom exceptions**: `AgentError`, `DataSourceError`, `GovernanceToolError`, `TicketingError`, `KnowledgeBaseError`, `OrchestratorError`
- **`safe_execute()`** — runs a function, returns fallback on exception


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. setup_logger — Dual Console + JSON File

In [ ]:
from core.logging_utils import setup_logger

logger = setup_logger("demo.notebook")
logger.info("Hello from notebook!")
logger.warning("This is a warning")
logger.error("This is an error")
print("\nlog written to logs/copilot.log")

In [ ]:
# Read last few lines of the JSON log
import json
try:
    with open("logs/copilot.log") as f:
        lines = f.readlines()
    for line in lines[-3:]:
        entry = json.loads(line)
        print(f"[{entry['level']}] {entry['message']}  (fn={entry['function']})")
except FileNotFoundError:
    print("Log file not found — check the logs/ directory path")

## 2. Custom Exception Hierarchy

In [ ]:
from core.logging_utils import (
    AgentError, DataSourceError, GovernanceToolError,
    TicketingError, KnowledgeBaseError, OrchestratorError
)

# All subclass AgentError (which subclasses Exception)
print("Inheritance chain:")
for cls in [DataSourceError, GovernanceToolError, TicketingError, KnowledgeBaseError]:
    print(f"  {cls.__name__} → {cls.__bases__[0].__name__} → Exception")

# Instantiate and inspect
try:
    raise DataSourceError("Databricks unreachable", agent_name="information_agent", recoverable=True)
except DataSourceError as e:
    print(f"\nCaught: {type(e).__name__}: {e}")
    print(f"  agent_name  : {e.agent_name}")
    print(f"  recoverable : {e.recoverable}")

## 3. @with_retry decorator

In [ ]:
from core.logging_utils import with_retry

attempt_count = 0

@with_retry(max_retries=3, delay_seconds=0.1, backoff=2.0)
def flaky_function():
    global attempt_count
    attempt_count += 1
    print(f"  Attempt {attempt_count}...")
    if attempt_count < 3:
        raise ConnectionError("Simulated connection error")
    return "Success!"

attempt_count = 0
result = flaky_function()
print("Result:", result)
print("Took", attempt_count, "attempts")

## 4. @timed decorator

In [ ]:
import time
from core.logging_utils import timed

@timed(agent_name="my_agent")
def slow_function():
    time.sleep(0.1)
    return 42

result = slow_function()
print("Returned:", result)  # duration logged automatically

## 5. safe_execute — never raises

In [ ]:
from core.logging_utils import safe_execute

# Normal case
result = safe_execute(lambda: 1 + 1)
print("Normal result:", result)

# Failing case — returns fallback
result = safe_execute(lambda: 1 / 0, fallback="fallback_value", log_error=False)
print("Fallback result:", result)

## 6. JSON log format inspection

In [ ]:
# Show full structure of a JSON log entry
logger2 = setup_logger("format.demo")
logger2.info("Structured log entry", extra={"agent": "info_agent", "query_id": "abc123", "duration_ms": 42.7})

with open("logs/copilot.log") as f:
    lines = f.readlines()
last = json.loads(lines[-1])
print("JSON log fields:")
for k, v in last.items():
    print(f"  {k:20s}: {v}")